In [ ]:
!pip install -q pybullet Box2D gym==0.25.2
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.3 MB/s eta 0:00:00
Fri Oct  2 19:24:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W 

In [ ]:
import os
import numpy as np
import torch as T
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.distributions import Normal

import warnings
warnings.filterwarnings('ignore')
from gym.wrappers import RecordVideo
import gym
import pybullet_envs
np.bool8 = np.bool_
from tqdm.notebook import trange
from IPython.display import Video

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
def plot_learning_curve(x, filename, save_plot=True):

    avg_x = [np.mean(x[max(0, i - 99):i + 1]) for i in range(len(x))]
    plt.figure(dpi=200)
    plt.title('Learning Curve')
    plt.plot(range(len(x)), x, label='score', alpha=0.3)
    plt.plot(range(len(avg_x)), avg_x, label='average score')
    plt.xlabel('Episode')
    plt.ylabel('Score')
    plt.legend()
    plt.grid()
    if save_plot:
        plt.savefig(filename + '.png')
    plt.show()

This class implements a **Replay Buffer** to store and sample transitions of the form $(s_t, a_t, r_t, s_{t+1}, d_t)$ to break correlation in updates for stability in mini-batch stochastic gradient descent.


In [ ]:
class ReplayBuffer:
    def __init__(self, buffer_size, state_dims, action_dims):
        self.buffer_size = buffer_size
        self.ptr = 0
        self.is_full = False
        if isinstance(state_dims, int):
            state_dims = (state_dims,)

        self.states = np.zeros((self.buffer_size, *state_dims), dtype=np.float32)
        self.states_ = np.zeros((self.buffer_size, *state_dims), dtype=np.float32)
        self.actions = np.zeros((self.buffer_size, action_dims), dtype=np.float32)
        self.rewards = np.zeros(self.buffer_size, dtype=np.float32)
        self.dones = np.zeros(self.buffer_size, dtype=bool)

    def store_transition(self, state, action, reward, state_, done):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.states_[self.ptr] = state_
        self.dones[self.ptr] = done

        self.ptr += 1
        if self.ptr == self.buffer_size:
            self.is_full = True
            self.ptr = 0


    def load_batch(self, batch_size):
        max_size = self.buffer_size if self.is_full else self.ptr

        batch_indices = np.random.choice(max_size, batch_size, replace=False)

        states = self.states[batch_indices]
        actions = self.actions[batch_indices]
        rewards = self.rewards[batch_indices]
        states_ = self.states_[batch_indices]
        dones = self.dones[batch_indices]

        return states, actions, rewards, states_, dones

## 🧱 Neural Networks

This cell defines three core neural networks used in SAC:

- **Critic Network:**
Estimates the **Q-value function** $ Q(s, a) $. Two critics are used to mitigate overestimation bias.

- **Value Network:**
Estimates the **state value function** $ V(s) $, used to train the actor and as a baseline.

- **Actor Network:**
Outputs the **mean** and **standard deviation** for a Gaussian policy
$
\pi(a|s) = \mathcal{N}(\mu(s), \sigma(s))
$.

In [ ]:
class Critic(nn.Module):
    def __init__(self, lr, state_dims, action_dims, fc1_dims, fc2_dims, name='Critic', ckpt_dir='tmp'):
        super(Critic, self).__init__()

        self.state_dims = state_dims
        self.action_dims = action_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(self.ckpt_dir, self.name + '_sac')
        self.fc1 = nn.Linear(self.state_dims + self.action_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.q = nn.Linear(self.fc2_dims, 1)
        self.optimizer = T.optim.Adam(self.parameters(), lr=lr)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state, action):
        action_value = F.relu(self.fc1(T.cat([state, action], dim=1)))
        action_value = F.relu(self.fc2(action_value))
        q = self.q(action_value)
        return q

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))


class Actor(nn.Module):
    def __init__(self, lr, state_dims, action_dims, fc1_dims, fc2_dims, max_action, reparam_noise,
                 name='Actor', ckpt_dir='tmp'):
        super(Actor, self).__init__()

        self.state_dims = state_dims
        self.action_dims = action_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.max_action = max_action
        self.reparam_noise = reparam_noise
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(self.ckpt_dir, self.name + '_sac')
        self.fc1 = nn.Linear(self.state_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.mu = nn.Linear(self.fc2_dims, self.action_dims)
        self.sigma = nn.Linear(self.fc2_dims, self.action_dims)

        self.optimizer = T.optim.Adam(self.parameters(), lr=lr)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)


    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mu = self.mu(x)
        sigma = self.sigma(x)
        sigma = T.clamp(sigma, min=self.reparam_noise, max=1)
        return mu, sigma

    def sample_normal(self, state, reparameterize=True):
        mu, sigma = self.forward(state)
        probabilities = Normal(mu, sigma)

        if reparameterize:
            actions = probabilities.rsample()
        else:
            actions = probabilities.sample()

        tanh_actions = T.tanh(actions)
        action = tanh_actions * self.max_action
        log_probs = probabilities.log_prob(actions)
        log_probs = log_probs - T.log(1 - tanh_actions.pow(2) + self.reparam_noise)
        log_probs = log_probs.sum(dim=1, keepdim=True)

        return action, log_probs

    def deterministic_action(self, state):
        mu, _ = self.forward(state)
        return T.tanh(mu) * self.max_action

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))


class Value(nn.Module):
    def __init__(self, lr, state_dims, fc1_dims, fc2_dims, name='Value', ckpt_dir='tmp'):
        super(Value, self).__init__()

        self.state_dims = state_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(self.ckpt_dir, self.name + '_sac')
        self.fc1 = nn.Linear(self.state_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.v = nn.Linear(self.fc2_dims, 1)
        self.optimizer = T.optim.Adam(self.parameters(), lr=lr)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)


    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        v = self.v(x)
        return v

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))

## 🤖 Agent Implementation

This class encapsulates the full logic of the SAC agent. In general, the learning process uses entropy-regularized policy gradients $J_\pi = \mathbb{E}_{s_t \sim D, a_t \sim \pi} \left[ \alpha \log(\pi(a_t|s_t)) - Q(s_t, a_t) \right]$ with soft target updates $\theta_{\text{target}} \leftarrow \tau \theta + (1 - \tau)\theta_{\text{target}}$.

In [ ]:
class Agent:
    def __init__(self, gamma, lr_actor, lr_critic, temperature, state_dims, action_dims, max_action,
                 fc1_dim, fc2_dim, memory_size, batch_size, tau, update_period, reward_scale, warmup,
                 reparam_noise_lim, name, ckpt_dir='tmp'):

        self.gamma = gamma
        self.lr_actor = lr_actor
        self.lr_critic = lr_critic
        self.alpha = temperature  # entropy temperature (separate from the learning rates!)
        self.state_dims = state_dims
        self.action_dims = action_dims
        self.max_action = max_action
        self.fc1_dim = fc1_dim
        self.fc2_dim = fc2_dim
        self.memory_size = memory_size
        self.batch_size = batch_size
        self.tau = tau
        self.update_period = update_period
        self.reward_scale = reward_scale
        self.warmup = warmup
        self.reparam_noise_lim = reparam_noise_lim
        self.ckpt_dir = ckpt_dir

        model_name = f'{name}__' \
                     f'gamma_{gamma}__' \
                     f'lra_{lr_actor}__' \
                     f'lrc_{lr_critic}__' \
                     f'temp_{temperature}__' \
                     f'fc1_{fc1_dim}__' \
                     f'fc2_{fc2_dim}__' \
                     f'bs_{batch_size}__' \
                     f'buffer_{memory_size}__' \
                     f'update_period_{update_period}__' \
                     f'tau_{tau}__'

        self.model_name = model_name
        self.learn_iter = 0
        self.total_steps = 0
        self.full_path = os.path.join(self.ckpt_dir, self.model_name)

        self.replay_buffer = ReplayBuffer(self.memory_size, self.state_dims, self.action_dims)

        self.actor = Actor(self.lr_actor, self.state_dims, self.action_dims, self.fc1_dim, self.fc2_dim,
                           self.max_action, self.reparam_noise_lim, name='Actor', ckpt_dir=self.ckpt_dir)
        self.critic_1 = Critic(self.lr_critic, self.state_dims, self.action_dims, self.fc1_dim, self.fc2_dim,
                               name='Critic1', ckpt_dir=self.ckpt_dir)
        self.critic_2 = Critic(self.lr_critic, self.state_dims, self.action_dims, self.fc1_dim, self.fc2_dim,
                               name='Critic2', ckpt_dir=self.ckpt_dir)

        self.value = Value(self.lr_critic, self.state_dims, self.fc1_dim, self.fc2_dim,
                           name='Value', ckpt_dir=self.ckpt_dir)
        self.target_value = Value(self.lr_critic, self.state_dims, self.fc1_dim, self.fc2_dim,
                                  name='TargetValue', ckpt_dir=self.ckpt_dir)

        self.target_value.load_state_dict(self.value.state_dict())

    def choose_action(self, state, deterministic=False, reparameterize=False):
        # purely random exploration during warmup
        if not deterministic and self.total_steps < self.warmup:
            return np.random.uniform(-self.max_action, self.max_action, size=self.action_dims).astype(np.float32)

        with T.no_grad():
            state = T.tensor(state, dtype=T.float).to(self.actor.device).unsqueeze(0)
            if deterministic:
                action = self.actor.deterministic_action(state)
            else:
                action, _ = self.actor.sample_normal(state, reparameterize=False)
        return action.cpu().numpy()[0]

    def store_transition(self, state, action, reward, state_, done):
        self.total_steps += 1
        self.replay_buffer.store_transition(state, action, reward, state_, done)

    def load_batch(self):
        states, actions, rewards, states_, done = \
        self.replay_buffer.load_batch(self.batch_size)
        return states, actions, rewards, states_, done

    def update_parameters(self, tau=None):
        if tau is None:
            tau = self.tau

        for target_param, param in zip(
            self.target_value.parameters(),
            self.value.parameters()
        ):
            target_param.data.copy_(
                tau * param.data +
                (1 - tau) * target_param.data
            )


    def save_model(self):
        print('... saving checkpoint ...')
        self.actor.save_checkpoint()
        self.critic_1.save_checkpoint()
        self.critic_2.save_checkpoint()
        self.value.save_checkpoint()
        self.target_value.save_checkpoint()

    def load_model(self, gpu_to_cpu=False):
        print('... loading checkpoint ...')
        self.actor.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.critic_1.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.critic_2.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.value.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.target_value.load_checkpoint(gpu_to_cpu=gpu_to_cpu)

    def learn(self):

        if self.total_steps < self.warmup:
            return
        if not self.replay_buffer.is_full and self.replay_buffer.ptr < self.batch_size:
            return

        states, actions, rewards, states_, done = self.load_batch()

        states = T.tensor(states, dtype=T.float).to(self.value.device)
        actions = T.tensor(actions, dtype=T.float).to(self.value.device)
        rewards = T.tensor(rewards, dtype=T.float).to(self.value.device)
        states_ = T.tensor(states_, dtype=T.float).to(self.value.device)
        done = T.tensor(done, dtype=T.float).to(self.value.device)
        actions_pi, log_probs = self.actor.sample_normal(states, reparameterize=False)

        q1 = self.critic_1(states, actions_pi)
        q2 = self.critic_2(states, actions_pi)

        q_min = T.min(q1, q2)

        value_target = (q_min - self.alpha * log_probs).detach()

        value = self.value(states)
        value_loss = 0.5 * F.mse_loss(value, value_target)

        self.value.optimizer.zero_grad()
        value_loss.backward()
        self.value.optimizer.step()
        actions_pi, log_probs = self.actor.sample_normal(states, reparameterize=True)
        q1_pi = self.critic_1(states, actions_pi)
        q2_pi = self.critic_2(states, actions_pi)
        min_q_pi = T.min(q1_pi, q2_pi)
        actor_loss = (self.alpha * log_probs - min_q_pi).mean()

        self.actor.optimizer.zero_grad()
        actor_loss.backward()
        self.actor.optimizer.step()

        with T.no_grad():

            rewards = rewards.unsqueeze(1)
            done = done.unsqueeze(1)
            q_target = self.reward_scale * rewards + self.gamma * self.target_value(states_) * (1 - done)
        q1_old = self.critic_1(states, actions)
        q2_old = self.critic_2(states, actions)
        q1_loss = 0.5 * F.mse_loss(q1_old, q_target)
        q2_loss = 0.5 * F.mse_loss(q2_old, q_target)
        self.critic_1.optimizer.zero_grad()
        q1_loss.backward()
        self.critic_1.optimizer.step()

        self.critic_2.optimizer.zero_grad()
        q2_loss.backward()
        self.critic_2.optimizer.step()

        self.learn_iter += 1
        if self.learn_iter % self.update_period == 0:
            self.update_parameters()

## ⚙️ Training Configuration

Set up your training parameters. `HalfCheetahBulletEnv-v0` is a continuous control task where the agent must learn to run using articulated legs.


In [ ]:
env_name = 'HalfCheetahBulletEnv-v0'
dir = 'tmp'
n_games = 250
gamma = 0.99
lr_actor = 3e-4
lr_critic = 3e-4
temperature = 1.0
fc1_dim = 256
fc2_dim = 256
memory_size = 1000000
batch_size = 256
tau = 0.005
update_period = 2
reward_scale = 2.
warmup = 1000
reparam_noise_lim = 1e-6
record_video = True

## 🚀 Training Loop

For each episode, interact with the environment to collect transitions, then update the SAC networks and save the best model.

After training, a learning curve is plotted to visualize convergence and performance stability.

In [ ]:
env = gym.make(env_name)
dir_path = os.path.join(dir, env_name)
os.makedirs(dir_path, exist_ok=True)

if record_video:
    env = RecordVideo(env, video_folder=os.path.join(dir_path, 'videos'),
                      episode_trigger=lambda ep: ep == n_games - 1)

state_dims = env.observation_space.shape[0]
action_dims = env.action_space.shape[0]
max_action = float(env.action_space.high[0])
agent = Agent(gamma=gamma, lr_actor=lr_actor, lr_critic=lr_critic, temperature=temperature,
              state_dims=state_dims, action_dims=action_dims, max_action=max_action,
              fc1_dim=fc1_dim, fc2_dim=fc2_dim, memory_size=memory_size, batch_size=batch_size,
              tau=tau, update_period=update_period, reward_scale=reward_scale,
              warmup=warmup, reparam_noise_lim=reparam_noise_lim, name='SAC_Cheetah',
              ckpt_dir=dir_path)

best_score = -np.inf
score_history = []
for game in trange(n_games):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    done = False
    score = 0

    while not done:
        action = agent.choose_action(state, deterministic=False, reparameterize=False)

        out = env.step(action)
        if len(out) == 5:
            state_, reward, terminated, truncated, info = out
            done = terminated or truncated
        else:
            state_, reward, done, info = out
            terminated = done and not info.get('TimeLimit.truncated', False)

        agent.store_transition(state, action, reward, state_, terminated)

        agent.learn()

        score += reward
        state = state_

    score_history.append(score)
    avg_score = np.mean(score_history[-100:])

    print(f'| Game: {game:6.0f} | Score: {score:10.2f} | Best score: {best_score:10.2f} | '
          f'Avg score {avg_score:10.2f} | Learning iter: {agent.learn_iter:10.0f} |')

    if avg_score > best_score:
        best_score = avg_score
        agent.save_model()

env.close()
plot_learning_curve(score_history, agent.full_path)

/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See ht

## 🎥 Visualize Agent Behavior

This is the last episode recorded video of the trained agent interacting with the environment in training process.

In [ ]:
Video(os.path.join(dir_path, 'videos', f'rl-video-episode-{n_games-1}.mp4'), embed=True, width=600)